# Calibrate Benchmark (L0→L1)

Measures eager open_l0 load cost, read_stride sensitivity, write cost, full vs decimated L1 size.

In [ ]:
from pathlib import Path

from panoseti_analysis.io.bench import BenchResult, stage_timer, summarize
from panoseti_analysis.paths import REPO_ROOT

# ── Configure ───────────────────────────────────────────────────────────────────────────────
L0_STORE = Path("/path/to/store.dp_img16.module_1.L0.zarr")  # replace
OUT_BASE = Path("/tmp/calibrate_bench")

results: list[BenchResult] = []

## §1 Baseline: full L0 load + full-res L1 write

In [ ]:
from panoseti_analysis.adapters.calibrate import run_calibrate

OUT_DIR = OUT_BASE / "baseline"
out_bytes = 0  # fill after run

with stage_timer("calibrate_baseline", bytes_in=0) as r:
    records = run_calibrate(L0_STORE, OUT_DIR)
    out_bytes = sum(f.stat().st_size for f in OUT_DIR.rglob("*") if f.is_file())

r.bytes_out = out_bytes
results.append(r)
print(summarize(results))

## §2 Decimated: read_stride=10

In [ ]:
OUT_DIR = OUT_BASE / "decimated"
out_bytes = 0  # fill after run

with stage_timer("calibrate_decimated", bytes_in=0) as r:
    records = run_calibrate(L0_STORE, OUT_DIR, read_stride=10)
    out_bytes = sum(f.stat().st_size for f in OUT_DIR.rglob("*") if f.is_file())

r.bytes_out = out_bytes
results.append(r)
print(summarize(results))

## §3 Summary

In [ ]:
print(summarize(results))